<a href="https://colab.research.google.com/github/AnjaliMenezes/open-bapm/blob/main/Adjusted_APM_for_WNBA_Version_1_(Based_on_Paper).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
install.packages(c("wehoop", "dplyr", "tidyr", "Matrix", "glmnet", "glue", "tibble"))

Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘snakecase’, ‘iterators’, ‘janitor’, ‘progressr’, ‘RcppParallel’, ‘stringdist’, ‘foreach’, ‘shape’, ‘RcppEigen’




In [2]:
#Load Data & Aggregate Stints
wnba_pbp <- wehoop::load_wnba_pbp()
wnba_player_box <- wehoop::load_wnba_player_box()

# Starters identification
starters <- wnba_player_box |>
  dplyr::filter(starter) |>
  dplyr::mutate(
    score_diff_final = (1 - 2 * (home_away == "away")) * (team_score - opponent_team_score)
  ) |>
  dplyr::select(game_id, home_away, athlete_id, score_diff_final)

# Substitutions extraction
substitutions <- wnba_pbp |>
  dplyr::filter(type_text == "Substitution") |>
  dplyr::mutate(
    time_start = 10 * (qtr - 1) + (9 - clock_minutes) + (60 - clock_seconds) / 60,
    time_remaining = end_game_seconds_remaining / 60,
    score_diff_start = home_score - away_score
  ) |>
  dplyr::select(game_id, time_start, time_remaining, score_diff_start, athlete_id_1, athlete_id_2)

player_on <- starters |>
  dplyr::group_by(game_id, home_away) |>
  dplyr::mutate(index = 1:dplyr::n(), time_start = 0, score_diff_start = 0, time_remaining = NA) |>
  dplyr::ungroup()

for (i in 1:nrow(substitutions)) {
  new_player_on <- player_on |>
    dplyr::filter(
      game_id == substitutions$game_id[i],
      athlete_id == substitutions$athlete_id_2[i]
    ) |>
    dplyr::slice(dplyr::n()) |>
    dplyr::mutate(
      athlete_id = substitutions$athlete_id_1[i],
      time_start = substitutions$time_start[i],
      time_remaining = substitutions$time_remaining[i],
      score_diff_start = substitutions$score_diff_start[i]
    )
  player_on <- dplyr::bind_rows(player_on, new_player_on)
}

lineups <- player_on |>
  dplyr::group_by(game_id, time_start, time_remaining, score_diff_start, score_diff_final, home_away, index) |>
  dplyr::summarize(athlete_id = athlete_id[dplyr::n()], .groups = "drop") |>
  dplyr::mutate(name = glue::glue("{home_away}_{index}")) |>
  dplyr::select(-home_away, -index) |>
  tidyr::pivot_wider(names_from = name, values_from = athlete_id) |>
  tidyr::fill(dplyr::matches("^away|^home"), .direction = "down")

athlete <- wnba_player_box |>
  dplyr::count(athlete_id, athlete_display_name, team_abbreviation) |>
  dplyr::group_by(athlete_id) |>
  dplyr::arrange(-n) |>
  dplyr::slice(1) |>
  dplyr::select(-n) |>
  dplyr::ungroup()

In [3]:
head(lineups)                                                                                     #
head(athlete)

game_id,time_start,time_remaining,score_diff_start,score_diff_final,away_1,away_2,away_3,away_4,away_5,home_1,home_2,home_3,home_4,home_5
<int>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
401856890,0.000000,NA,0,31,4433635,4684384,2490553,3922628,4433514,2998928,2999101,2593770,4038379,5345320
401856890,4.466667,35.38333,10,31,4703609,4684384,2490553,3922628,4433514,2998928,2999101,2593770,4038379,5345320
401856890,4.466667,35.53333,10,31,4703609,4684384,2490553,3922628,4433514,2998928,2327695,4337216,4038379,5345320
401856890,4.800000,35.13333,10,31,4703609,4684384,2490553,3922628,4433514,2998928,2327695,4337216,2566110,5345320
401856890,5.800000,33.91667,15,31,4703609,4398966,2490553,3922628,4433514,2998928,2327695,4337216,2566110,5345320
401856890,6.266667,33.73333,15,31,4703609,4398966,2490553,3922628,4433514,2998928,2327695,4337216,2566110,4038379


athlete_id,athlete_display_name,team_abbreviation
<int>,<chr>,<chr>
869,DeWanna Bonner,PHX
887,Sami Whitcomb,PHX
924,Alysha Clark,DAL
981,Courtney Vandersloot,CHI
1054,Tiffany Hayes,GS
1068,Nneka Ogwumike,LA


In [4]:

data <- lineups |>
  dplyr::group_by(game_id) |>
  dplyr::mutate(
    # Stint duration in minutes
    minutes = dplyr::coalesce(dplyr::lead(time_start, 1) - time_start, time_remaining),
    # Raw point differential for the stint (Home perspective)
    score_diff = dplyr::coalesce(dplyr::lead(score_diff_start, 1), score_diff_final) - score_diff_start,
    # Possessions: Estimated at 2 per minute (~80 pace), or join with play event counts
    possessions = minutes * 2,
    # Target variable (t_i): Score differential per 100 possessions
    score_diff_100 = ifelse(possessions > 0, 100 * (score_diff / possessions), 0)
  ) |>
  dplyr::ungroup() |>
  dplyr::filter(possessions > 0)


data_long <- data |>
  dplyr::mutate(stint = 1:dplyr::n()) |>
  dplyr::select(game_id, stint, minutes, possessions, score_diff, score_diff_100, dplyr::matches("^away|^home")) |>
  tidyr::pivot_longer(
    cols = c(dplyr::matches("^away|^home")),
    names_to = "name",
    values_to = "athlete_id"
  )

In [5]:
y <- data$score_diff_100
w <- data$possessions

unique_athletes <- sort(unique(data_long$athlete_id))
n_players <- length(unique_athletes)

# HCA Column: Index 1 set to +1
hca_data <- tibble::tibble(
  row = 1:nrow(data),
  column = 1,
  value = 1
)

# Player Columns: Shifted by +1 (Indices 2 to N+1)
player_data <- data_long |>
  dplyr::mutate(
    row = stint,
    column = match(athlete_id, unique_athletes) + 1,
    value = ifelse(substring(name, 1, 4) == "home", 1, -1)
  )

# Build Sparse Design Matrix x_i
x_mat <- Matrix::sparseMatrix(
  i = c(hca_data$row, player_data$row),
  j = c(hca_data$column, player_data$column),
  x = c(hca_data$value, player_data$value),
  dims = c(nrow(data), n_players + 1)
)

set.seed(42)
model_formula1_rapm <- glmnet::cv.glmnet(
  x = x_mat,
  y = y,
  weights = w,
  alpha = 0,
  intercept = FALSE,
  standardize = FALSE,
  nfolds = 10
)

In [6]:
w_hat <- coef(model_formula1_rapm, s = "lambda.min")[-1, 1]

hca_100 <- w_hat[1]

athlete_coef_rapm <- tibble::tibble(
  athlete_id = unique_athletes,
  rapm_100 = w_hat[2:(n_players + 1)]
)

athlete_summary <- data_long |>
  dplyr::group_by(athlete_id) |>
  dplyr::summarize(
    possessions = sum(possessions),
    raw_pm_100 = 100 * sum((1 - 2 * grepl("away", name)) * score_diff) / sum(possessions),
    .groups = "drop"
  )

final_rapm_leaderboard <- athlete_summary |>
  dplyr::inner_join(athlete_coef_rapm, by = "athlete_id") |>
  dplyr::left_join(athlete, by = "athlete_id") |>
  dplyr::filter(possessions >= 250) |>
  dplyr::select(athlete_display_name, team_abbreviation, possessions, raw_pm_100, rapm_100) |>
  dplyr::arrange(-rapm_100)

head(final_rapm_leaderboard, 10)

athlete_display_name,team_abbreviation,possessions,raw_pm_100,rapm_100
<chr>,<chr>,<dbl>,<dbl>,<dbl>
A'ja Wilson,LV,2404.853,12.266860,4.518895
Jackie Young,LV,2632.857,10.558873,3.992810
Veronica Burton,GS,2272.017,12.103784,3.838773
Natasha Howard,MIN,2391.427,12.419365,3.222421
Olivia Miles,MIN,2394.467,11.568338,3.037677
Allisha Gray,ATL,2686.250,9.604467,3.035149
Kaila Charles,GS,1570.613,13.879928,2.848350
Aliyah Boston,IND,2024.723,10.371787,2.832365
Kelsey Mitchell,IND,2710.947,9.037433,2.824890
